# Part 2: Customer Churn Prediction

## Goal

This exercise estimates the probability that a customer will churn. Customers with predicted probabilities of at least 0.50 are labeled at risk.

In [4]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Data source/reference: IBM Telco Customer Churn dataset
# https://www.kaggle.com/datasets/blastchar/telco-customer-churn
# The published dataset inspired the feature design. This reproducible
# educational version contains 500 records with similar business variables.

rng = np.random.default_rng(7)
n = 500
age = rng.integers(18, 76, n)
monthly_usage = np.round(rng.normal(42, 16, n).clip(5, 100), 1)
purchase_amount = np.round(rng.normal(95, 45, n).clip(10, 300), 2)
service_calls = rng.poisson(2.2, n).clip(0, 10)
months_with_company = rng.integers(1, 73, n)
region = rng.choice(["West", "Central", "East", "South"], n)
region_effect = {"West": 0.10, "Central": 0.00, "East": -0.08, "South": 0.05}
logit = (0.9 - 0.025 * monthly_usage - 0.012 * purchase_amount
         + 0.33 * service_calls - 0.018 * months_with_company
         + 0.006 * (age - 40)
         + np.array([region_effect[x] for x in region])
         + rng.normal(0, 0.45, n))
churn_probability = 1 / (1 + np.exp(-logit))
churn = rng.binomial(1, churn_probability)

churn_data = pd.DataFrame({
    "age": age,
    "monthly_usage_hours": monthly_usage,
    "purchase_amount": purchase_amount,
    "customer_service_calls": service_calls,
    "months_with_company": months_with_company,
    "region": region,
    "churn": churn,
})
churn_data.to_csv("customer_churn_realistic.csv", index=False)
churn_data = pd.read_csv("customer_churn_realistic.csv")
print(f"Records loaded: {len(churn_data)}")
print(churn_data.head())

Records loaded: 500
   age  monthly_usage_hours  purchase_amount  customer_service_calls  \
0   72                  6.8           126.76                       2   
1   54                 30.9            64.56                       2   
2   57                 10.5           159.92                       1   
3   70                  5.0            92.47                       3   
4   51                 33.5            91.90                       4   

   months_with_company region  churn  
0                   20  South      1  
1                    4   East      1  
2                    5   East      0  
3                   31   East      0  
4                   35   East      0  


In [5]:
X = churn_data.drop(columns="churn")
y = churn_data["churn"]
numerical_features = ["age", "monthly_usage_hours", "purchase_amount", "customer_service_calls", "months_with_company"]
categorical_features = ["region"]

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numerical_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical_features),
])
model = Pipeline([
    ("preprocessor", preprocessor),
    ("logistic_regression", LogisticRegression(max_iter=1000, random_state=42)),
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
model.fit(X_train, y_train)

predicted_classes = model.predict(X_test)
predicted_probabilities = model.predict_proba(X_test)[:, 1]
print(f"Accuracy: {accuracy_score(y_test, predicted_classes):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, predicted_probabilities):.3f}")
print(classification_report(y_test, predicted_classes, zero_division=0))

Accuracy: 0.760
ROC-AUC: 0.695
              precision    recall  f1-score   support

           0       0.78      0.95      0.85        73
           1       0.64      0.26      0.37        27

    accuracy                           0.76       100
   macro avg       0.71      0.60      0.61       100
weighted avg       0.74      0.76      0.72       100



In [6]:
new_customer = pd.DataFrame({
    "age": [29],
    "monthly_usage_hours": [18],
    "purchase_amount": [35],
    "customer_service_calls": [5],
    "months_with_company": [8],
    "region": ["West"],
})
probability = model.predict_proba(new_customer)[0, 1]
classification = "At risk of churning" if probability >= 0.50 else "Not currently classified as at risk"
print(f"New customer churn probability: {probability:.1%}")
print(f"Classification: {classification}")

feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["logistic_regression"].coef_[0]
coefficient_table = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
print("\nLogistic regression coefficients:")
print(coefficient_table.to_string(index=False))

New customer churn probability: 85.6%
Classification: At risk of churning

Logistic regression coefficients:
                        feature  coefficient
                   numeric__age     0.150092
   numeric__monthly_usage_hours    -0.371692
       numeric__purchase_amount    -0.661902
numeric__customer_service_calls     0.531035
   numeric__months_with_company    -0.416152
       categorical__region_East    -0.554844
      categorical__region_South    -0.415094
       categorical__region_West    -0.087810


### Interpretation

The churn probability is the model's estimated chance that a customer will leave. A 0.50 threshold converts that probability into a simple yes-or-no risk label. Businesses can use high-risk scores to prioritize retention calls, service recovery, discounts, or product education. The model supports decisions but should not be treated as a guarantee.